In [ ]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Core.CalcEngine import Engine, ParallelEngine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.Factor.BasicOperator import rename
import QuantStudio.Factor.FactorOperator as fo
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Risk.HDF5RDB import HDF5FRDB
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime

FDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()
RDB = HDF5FRDB(args={"MainDir": "../data/Risk"}).connect()

# Bias Test

In [ ]:
from QuantStudio.BackTest.Risk.BiasTest import BiasTest

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 31)# 数据起止时间
TestStartDT, TestEndDT = dt.datetime(2025, 2, 28), EndDT# 测试起止时间

FT = FDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

DTs = FT.getDateTime(ifactor_name="close", start_dt=StartDT, end_dt=EndDT)
SecionIDs = IDs = FT.getID(ifactor_name="close")

# 再平衡时点序列
BalanceDTs = getMonthLastDateTime(DTs)# 月末

Price = FDB.getTable("stock_cn_day_bar").getFactor("close")
IfListed = FDB.getTable("stock_cn_status").getFactor("if_listed")
Industry = FDB.getTable("stock_cn_industry").getFactor("industry")
Mask = (IfListed == 1)

RT = RDB.getTable("demo_risk_table")

# Bias Test
BiasTestNode = BiasTest(
    descriptor_ids=SectionIDs,
    price=Price,
    risk_table=RT,
    mask=Mask,
    industry=Industry,
    args={"RebalanceDTs": BalanceDTs, "RandomNums": [5], "IndustryList": ["TMT", "Ind", "Fin"], "RollingAvgPeriod": 1, "GenReport": True}
)

NodeList = [BiasTestNode]
Report = BTReport(bt_node_list=NodeList)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": "../data/Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context, fwd_data_list=[DTLocalContext(DTs=TestDTs)], init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

display(HTML(Rslt[0]["Report"]))

==========历史回测==========
1. 初始化



  0% (0 of 566) |                        | Elapsed Time: 0:00:00 ETA:  --:--:--

耗时 : 8.61
2. 循环计算


100% (566 of 566) |######################| Elapsed Time: 0:01:17 Time:  0:01:17


耗时 : 77.61
3. 结果生成
耗时 : 0.21
总耗时 : 86.42


,策略组合收益,基准组合收益,主动资产配置组合收益,主动个券选择组合收益,主动资产配置超额收益,主动个券选择超额收益,交互作用超额收益,总超额收益
现金,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Banks,6.58%,3.71%,6.94%,3.52%,3.23%,-0.19%,-0.17%,2.87%
Real Estate,0.87%,1.29%,0.69%,1.71%,-0.59%,0.42%,-0.25%,-0.42%
Industrial Conglomerates,0.00%,-0.08%,0.00%,0.00%,0.08%,0.08%,-0.08%,0.08%
Health,0.61%,1.29%,0.30%,1.58%,-1.00%,0.29%,0.03%,-0.68%
Commercial and Professional Services,0.00%,-0.48%,0.00%,0.00%,0.48%,0.48%,-0.48%,0.48%
Hotels Restaurants and Leisure,0.32%,0.58%,0.27%,0.25%,-0.31%,-0.32%,0.37%,-0.25%
Industrial Machinery,0.00%,0.49%,0.44%,-0.04%,-0.05%,-0.53%,0.09%,-0.49%
Construction and Engineering,-0.37%,-0.33%,-0.59%,-0.20%,-0.26%,0.14%,0.08%,-0.04%
Construction Materials,0.27%,0.42%,0.24%,0.25%,-0.18%,-0.16%,0.20%,-0.15%
